# Leakage-Safe Training Visualizations

This notebook reads the completed training logs and displays figures inline. It intentionally contains no `savefig` or export calls. Review each visual first; saving can be added later after figure design is approved.

## 1. Setup and load training logs

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11})

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'results' / 'leakage_safe').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / 'results' / 'leakage_safe'
epoch_log_path = RESULTS_DIR / 'training_log.csv'
batch_log_path = RESULTS_DIR / 'training_batch_log.csv'

epoch_df = pd.read_csv(epoch_log_path)
batch_df = pd.read_csv(batch_log_path)
print(f'Epoch rows: {len(epoch_df)} | Batch rows: {len(batch_df):,}')
display(epoch_df.head())

## 2. Data-integrity summary

In [ ]:
expected_columns = {
    'epoch', 'train_loss', 'val_loss', 'val_dice_wt',
    'val_dice_tc', 'val_dice_et', 'learning_rate'
}
assert expected_columns.issubset(epoch_df.columns)
assert epoch_df['epoch'].is_unique
assert epoch_df['epoch'].is_monotonic_increasing

summary = pd.DataFrame({
    'start': epoch_df.iloc[0],
    'final': epoch_df.iloc[-1],
    'minimum': epoch_df.min(numeric_only=True),
    'maximum': epoch_df.max(numeric_only=True),
})
display(summary.loc[['epoch', 'train_loss', 'val_loss', 'val_dice_wt', 'val_dice_tc', 'val_dice_et', 'learning_rate']])

## 3. Epoch-level training and validation loss

In [ ]:
fig, ax = plt.subplots()
ax.plot(epoch_df['epoch'], epoch_df['train_loss'], label='Training loss', linewidth=2)
ax.plot(epoch_df['epoch'], epoch_df['val_loss'], label='Validation loss', linewidth=2)
ax.set(title='Training and Validation Loss', xlabel='Epoch', ylabel='Dice + CE loss')
ax.legend()
plt.show()

## 4. Batch-level training loss with smoothing

In [ ]:
smoothing_window = 250
batch_df['smoothed_loss'] = batch_df['batch_loss'].rolling(smoothing_window, min_periods=1).mean()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(batch_df['global_step'], batch_df['batch_loss'], alpha=0.12, linewidth=0.6, label='Batch loss')
ax.plot(batch_df['global_step'], batch_df['smoothed_loss'], linewidth=2, label=f'{smoothing_window}-step mean')
ax.set(title='Batch-Level Training Loss', xlabel='Global step', ylabel='Dice + CE loss')
ax.legend()
plt.show()

## 5. Validation Dice by BraTS region

In [ ]:
fig, ax = plt.subplots()
ax.plot(epoch_df['epoch'], epoch_df['val_dice_wt'], label='WT', linewidth=2)
ax.plot(epoch_df['epoch'], epoch_df['val_dice_tc'], label='TC', linewidth=2)
ax.plot(epoch_df['epoch'], epoch_df['val_dice_et'], label='ET', linewidth=2)
ax.set(title='Validation Dice by Tumour Region', xlabel='Epoch', ylabel='Dice', ylim=(0, 1))
ax.legend()
plt.show()

## 6. Mean validation Dice and best-checkpoint epoch

In [ ]:
dice_columns = ['val_dice_wt', 'val_dice_tc', 'val_dice_et']
epoch_df['mean_val_dice'] = epoch_df[dice_columns].mean(axis=1)
best_index = epoch_df['mean_val_dice'].idxmax()
best_row = epoch_df.loc[best_index]

fig, ax = plt.subplots()
ax.plot(epoch_df['epoch'], epoch_df['mean_val_dice'], color='black', linewidth=2, label='Mean region Dice')
ax.scatter(best_row['epoch'], best_row['mean_val_dice'], color='crimson', s=70, zorder=3, label='Best checkpoint')
ax.axvline(best_row['epoch'], color='crimson', linestyle='--', alpha=0.6)
ax.set(title='Mean Validation Dice and Selected Checkpoint', xlabel='Epoch', ylabel='Mean Dice', ylim=(0, 1))
ax.legend()
plt.show()
display(best_row[['epoch', 'val_dice_wt', 'val_dice_tc', 'val_dice_et', 'mean_val_dice', 'val_loss']].to_frame('best'))

## 7. Generalization gap

In [ ]:
epoch_df['loss_gap'] = epoch_df['val_loss'] - epoch_df['train_loss']
fig, ax = plt.subplots()
ax.plot(epoch_df['epoch'], epoch_df['loss_gap'], color='purple', linewidth=2)
ax.axhline(0, color='black', linewidth=1)
ax.set(title='Validation–Training Loss Gap', xlabel='Epoch', ylabel='Loss gap')
plt.show()

## 8. Learning-rate history

In [ ]:
fig, ax = plt.subplots()
ax.plot(epoch_df['epoch'], epoch_df['learning_rate'], linewidth=2)
ax.set(title='Learning Rate', xlabel='Epoch', ylabel='Learning rate')
ax.ticklabel_format(style='plain', axis='y')
plt.show()

## 9. Compact review dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].plot(epoch_df['epoch'], epoch_df['train_loss'], label='Train')
axes[0, 0].plot(epoch_df['epoch'], epoch_df['val_loss'], label='Validation')
axes[0, 0].set_title('Loss'); axes[0, 0].legend()
for column, label in zip(dice_columns, ['WT', 'TC', 'ET']):
    axes[0, 1].plot(epoch_df['epoch'], epoch_df[column], label=label)
axes[0, 1].set_title('Validation Dice'); axes[0, 1].set_ylim(0, 1); axes[0, 1].legend()
axes[1, 0].plot(epoch_df['epoch'], epoch_df['mean_val_dice'], color='black')
axes[1, 0].axvline(best_row['epoch'], color='crimson', linestyle='--')
axes[1, 0].set_title('Mean Dice and Best Epoch')
axes[1, 1].plot(epoch_df['epoch'], epoch_df['loss_gap'], color='purple')
axes[1, 1].axhline(0, color='black', linewidth=1)
axes[1, 1].set_title('Generalization Gap')
for ax in axes.flat:
    ax.set_xlabel('Epoch')
fig.tight_layout()
plt.show()